# Model Architecture Profiling

Understand the model's structure: parameter counts, computation flow,
hook points, capacity utilization, and summary.

## Why This Matters

Model architecture provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_architecture_profiling import (
    parameter_count_profile, computation_flow_profile,
    hook_point_inventory, model_capacity_utilization,
    model_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Model Summary

Architecture at a glance.

In [ ]:
summary = model_summary(model)
for k, v in summary.items():
    print(f"  {k}: {v}")

## Parameter Count Profile

Where are the model's parameters allocated?

In [ ]:
result = parameter_count_profile(model)
print(f"Total parameters: {result['total_parameters']:,}")
print(f"  Embedding: {result['embedding_params']:,} ({result['embed_fraction']:.1%})")
print(f"  Positional: {result['positional_params']:,}")
print(f"  Unembed: {result['unembed_params']:,}")
print(f"  Layers ({result['n_layers']}): {result['total_layer_params']:,} ({result['layer_fraction']:.1%})")
print(f"    Per layer: attn={result['per_layer_attn']:,}, mlp={result['per_layer_mlp']:,}, ln={result['per_layer_ln']:,}")

## Computation Flow Profile

Magnitude of computation at each stage.

In [ ]:
result = computation_flow_profile(model, tokens)
print(f"Embed norm: {result['embed_norm']:.4f}\n")
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 20)
    mlp_bar = '░' * int((1-p['attn_fraction']) * 20)
    print(f"  Layer {p['layer']}: attn={p['attn_output_norm']:.4f}, "
          f"mlp={p['mlp_output_norm']:.4f}, "
          f"resid={p['residual_norm']:.4f} {attn_bar}{mlp_bar}")

## Hook Point Inventory

All available hook points for caching and intervention.

In [ ]:
result = hook_point_inventory(model)
print(f"Total hooks: {result['total_hooks']}")
print(f"  Embed: {result['n_embed']}")
print(f"  Residual: {result['n_residual']}")
print(f"  Attention: {result['n_attention']}")
print(f"  MLP: {result['n_mlp']}\n")
print('First 10 hooks:')
for h in result['hook_points'][:10]:
    print(f"  [{h['type']:10s}] {h['name']}")

## Capacity Utilization

Effective rank of OV circuits and MLP weights.

In [ ]:
result = model_capacity_utilization(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean_OV_rank={p['mean_ov_rank']:.2f}, "
          f"MLP_rank={p['mlp_effective_rank']:.2f}")